# Week 8 — Pandas & NumPy
**Phase 2: Python for Data Analysis**

You write SQL against `dex.trades` every day. This week you learn to run the same logic **locally, in Python**, on any data you can get — CSVs, JSON from DefiLlama, API responses — without a query engine in between.

**What you'll build:** a DEX trade analysis toolkit — filtering, aggregating, joining, gas statistics, and TVL time series.

**Prerequisites:** Weeks 2–7, plus `blockchain_utils.py` from Week 5. Needs `pandas>=2.2` and `numpy`.

## 0. Setup

Two things first: install check, and importing your Week 5 toolkit. Notebooks in this folder reach back to Phase 1 with a relative path.

In [ ]:
# pip install pandas numpy   (already in your .venv if you ran the Week 1 setup)
import sys
sys.path.insert(0, "../../phase-1-python-fundamentals/week-05-functions-modules")

import numpy as np
import pandas as pd
from blockchain_utils import *          # your Week 5 toolkit
from sample_data import (make_dex_trades, make_gas_prices, make_tvl,
                         make_pair_info, make_wallet_labels)

pd.set_option("display.float_format", "{:,.2f}".format)   # readable numbers
pd.set_option("display.width", 120)
print("pandas", pd.__version__, "| numpy", np.__version__)

## 1. Why Pandas? The SQL mental model

A SQL engine and Pandas do the same job: **take a table, transform it, return a table.** The difference is where it runs and how you write it.

| | SQL (DuneSQL) | Pandas |
|---|---|---|
| Runs | On the provider's cluster | In your Python process, in memory |
| Data source | Tables someone indexed for you | Anything: CSV, JSON, API, SQL result |
| Logic | One declarative query | Step-by-step, inspectable at every line |
| Limit | Credits / query timeout | Your RAM (~ a few GB is comfortable) |
| Strength | Billions of rows | Fast iteration, ML, charts, custom logic |

**The practical workflow of a data engineer:** use SQL to reduce billions of rows to millions, then pull the result into Pandas for the part SQL is awkward at — feature engineering, joins against API data, plotting, models.

Pandas is built on **NumPy**, which stores numbers in compact typed arrays so maths runs in C instead of Python loops. Section 9 covers that layer directly.

## 2. The DataFrame is a table

A **DataFrame** is a table: named columns, each column holding one data type, and a row index. A single column is a **Series**.

```
SQL table  ->  DataFrame
column     ->  Series (one column, one dtype)
row        ->  a row label in the index
```

Let's load 14 days of synthetic DEX trades. The shape matches `dex.trades` on Dune: one row per swap.

In [ ]:
trades = make_dex_trades()
trades.head()          # SQL: SELECT * FROM trades LIMIT 5

### 2.1 First look: the five commands you run on every dataset

Before any analysis, you inspect. These are your `DESCRIBE`, `COUNT(*)`, and null-check, all in one:

In [ ]:
print("shape:", trades.shape)             # (rows, columns)   SQL: SELECT COUNT(*)
print()
trades.info()                             # dtypes + non-null counts  SQL: DESCRIBE

In [ ]:
trades.describe()      # count, mean, std, min, quartiles, max for numeric columns

In [ ]:
trades.isna().sum()    # NULLs per column.  SQL: COUNT(*) - COUNT(col) for each column

Notice `gas_price_gwei` has missing values. In SQL they're `NULL`; in Pandas they're `NaN` (a float "not a number"). Both **propagate**: `NaN + 5` is `NaN`, just like `NULL + 5` is `NULL`. Aggregations like `mean()` skip them by default, same as `AVG()`.

### 2.2 Series: one column

Selecting one column gives a Series. You can do maths on the whole thing at once — no loop.

In [ ]:
amounts = trades["amount_usd"]
print(type(amounts))
print("total volume :", fmt_usd(amounts.sum()))
print("median trade :", fmt_usd(amounts.median()))
print("biggest trade:", fmt_usd(amounts.max()))

Mean vs median is a classic on-chain trap. Trade sizes are heavy-tailed (a few whales dominate), so the mean is pulled far above what a typical trader does:

In [ ]:
print("mean  :", fmt_usd(amounts.mean()))
print("median:", fmt_usd(amounts.median()))

## 3. Loading and saving data

In Week 7 you read files with `open()` and `json`. Pandas wraps all of that.

| Format | Read | Write |
|---|---|---|
| CSV | `pd.read_csv(path)` | `df.to_csv(path, index=False)` |
| JSON | `pd.read_json(path)` | `df.to_json(path, orient="records")` |
| SQL | `pd.read_sql(query, conn)` | `df.to_sql(name, conn)` (Week 10) |
| Parquet | `pd.read_parquet(path)` | `df.to_parquet(path)` (Phase 5) |

The one habit that matters: **tell Pandas which columns are dates** when loading, otherwise they arrive as plain strings.

In [ ]:
trades.to_csv("dex_trades.csv", index=False)               # index=False: don't write the 0..n row numbers

raw = pd.read_csv("dex_trades.csv")
print("without parse_dates:", raw["block_time"].dtype)       # object (string)

df = pd.read_csv("dex_trades.csv", parse_dates=["block_time"])
print("with parse_dates   :", df["block_time"].dtype)        # datetime64

**From an API response.** DefiLlama and CoinGecko return a list of dicts (JSON). A list of dicts is already table-shaped:

In [ ]:
api_style_response = [
    {"name": "Aave",    "tvl": 12.4e9, "chain": "Ethereum"},
    {"name": "Lido",    "tvl": 26.1e9, "chain": "Ethereum"},
    {"name": "Uniswap", "tvl": 5.3e9,  "chain": "Ethereum"},
]
pd.DataFrame(api_style_response)

## 4. Selecting columns and rows (`SELECT`)

### 4.1 Columns

In [ ]:
df["pair"]                           # one column -> Series
df[["tx_hash", "pair", "amount_usd"]].head(3)   # several columns -> DataFrame (note the double brackets)

Double brackets matter: the inner `[...]` is a Python list of column names, the outer `[...]` is the selection operator. SQL equivalent: `SELECT tx_hash, pair, amount_usd`.

### 4.2 Rows: `loc` vs `iloc`

- `iloc` selects by **position** (row number, like array indexing)
- `loc` selects by **label** (index value or boolean mask) and can pick columns by name

Our index is just 0..999, so they look similar here, but they diverge as soon as you filter or sort.

In [ ]:
print(df.iloc[0])                      # first row, by position
print()
df.loc[0:2, ["pair", "amount_usd"]]    # rows labelled 0..2 (loc includes the end!), two columns

Gotcha: `iloc[0:3]` returns 3 rows (Python slicing, end excluded) but `loc[0:2]` also returns 3 rows (label slicing, end **included**). Easy to trip on.

## 5. Filtering (`WHERE`)

This is the most used operation in all of data analysis. The mechanism: build a **boolean Series** (True/False per row), then use it to keep only the True rows.

In [ ]:
is_big = df["amount_usd"] > 10_000
print(is_big.head())                   # True/False per row
print("big trades:", is_big.sum())      # True counts as 1

df[is_big].head(3)                      # SQL: WHERE amount_usd > 10000

### 5.1 Combining conditions: `&`, `|`, `~`

| SQL | Pandas |
|---|---|
| `AND` | `&` |
| `OR` | `\|` |
| `NOT` | `~` |

Two rules that will save you hours: **wrap each condition in parentheses** (operator precedence), and **never use Python's `and` / `or`** here. They don't work element-wise and raise an error.

In [ ]:
# WHERE protocol = 'Uniswap' AND side = 'buy' AND amount_usd > 5000
mask = (df["protocol"] == "Uniswap") & (df["side"] == "buy") & (df["amount_usd"] > 5000)
df[mask][["block_time", "pair", "amount_usd"]].head()

### 5.2 The rest of the `WHERE` toolbox

| SQL | Pandas |
|---|---|
| `IN ('Curve', 'Balancer')` | `df["protocol"].isin(["Curve", "Balancer"])` |
| `BETWEEN 1000 AND 5000` | `df["amount_usd"].between(1000, 5000)` |
| `IS NULL` | `df["gas_price_gwei"].isna()` |
| `IS NOT NULL` | `df["gas_price_gwei"].notna()` |
| `LIKE 'WETH%'` | `df["pair"].str.startswith("WETH")` |
| `LIKE '%ETH%'` | `df["pair"].str.contains("ETH")` |

In [ ]:
print("IN       :", df["protocol"].isin(["Curve", "Balancer"]).sum())
print("BETWEEN  :", df["amount_usd"].between(1_000, 5_000).sum())
print("IS NULL  :", df["gas_price_gwei"].isna().sum())
print("LIKE WETH%:", df["pair"].str.startswith("WETH").sum())

### 5.3 `.query()` — the readable alternative

For long filters, `.query()` reads almost like SQL. Use `@` to reference Python variables:

In [ ]:
min_size = 5_000
df.query("protocol == 'Curve' and amount_usd > @min_size").head(3)

## 6. Creating columns, sorting, limiting

### 6.1 New columns (computed `SELECT` expressions)

Assigning to a new column name creates it, computed across all rows at once (**vectorised**). Use `.assign()` to chain without mutating the original.

In [ ]:
df = df.assign(
    gas_cost_eth = df["gas_used"] * df["gas_price_gwei"] / 1e9,      # SQL: gas_used * gas_price / 1e9 AS gas_cost_eth
    trade_date   = df["block_time"].dt.date,                         # SQL: date(block_time)
    hour         = df["block_time"].dt.hour,                         # SQL: hour(block_time)
)
df[["tx_hash", "gas_used", "gas_price_gwei", "gas_cost_eth", "hour"]].head()

The `.dt` accessor exposes date parts (`.year`, `.month`, `.dayofweek`, `.hour`) and `.str` exposes string methods. Both work on a whole column.

### 6.2 Reusing your `blockchain_utils` with `.apply()`

Your Week 5 functions take single values. `.apply()` runs a function once per element/row. It's convenient but **slow** (a Python loop in disguise). Fine for 1000 rows; for millions, use vectorised maths instead.

In [ ]:
ETH_PRICE = 3_250.0

# Convenient: reuse your own function per row
df["gas_cost_usd_apply"] = df.apply(
    lambda r: gas_cost_usd(r["gas_used"], r["gas_price_gwei"], ETH_PRICE)
    if pd.notna(r["gas_price_gwei"]) else np.nan,
    axis=1,
)

# Fast: same maths, vectorised (what you'd do on 10M rows)
df["gas_cost_usd"] = df["gas_cost_eth"] * ETH_PRICE

# Same answer, different speed
print("identical:", np.allclose(df["gas_cost_usd"], df["gas_cost_usd_apply"], equal_nan=True))
df[["tx_hash", "gas_cost_usd"]].head(3)

In [ ]:
%timeit df.apply(lambda r: gas_cost_usd(r["gas_used"], r["gas_price_gwei"], ETH_PRICE) if pd.notna(r["gas_price_gwei"]) else np.nan, axis=1)
%timeit df["gas_cost_eth"] * ETH_PRICE

Expect the vectorised line to be **over a hundred times faster** (about 150x on the machine this was written on). This is the single most important performance idea in Pandas: *operate on whole columns, not row by row.*

### 6.3 Conditional columns: `CASE WHEN`

`np.where(condition, value_if_true, value_if_false)` is `CASE WHEN`. For more than two branches use `np.select`.

In [ ]:
df["size_bucket"] = np.select(
    [df["amount_usd"] >= 50_000, df["amount_usd"] >= 5_000, df["amount_usd"] >= 500],
    ["whale", "large", "medium"],
    default="retail",
)
# SQL: CASE WHEN amount_usd >= 50000 THEN 'whale' WHEN ... ELSE 'retail' END
df["size_bucket"].value_counts()      # SQL: SELECT size_bucket, COUNT(*) ... GROUP BY 1 ORDER BY 2 DESC

### 6.4 Sorting and limiting (`ORDER BY ... LIMIT`)

In [ ]:
# ORDER BY amount_usd DESC LIMIT 5
df.sort_values("amount_usd", ascending=False).head(5)[["tx_hash", "pair", "amount_usd"]]

In [ ]:
# Shortcut for top-N: nlargest / nsmallest
df.nlargest(3, "gas_cost_usd")[["tx_hash", "gas_cost_usd"]]

## 7. GroupBy (`GROUP BY`)

GroupBy follows **split → apply → combine**: split rows into groups by key, apply an aggregation to each, combine into a new table.

```sql
SELECT protocol, COUNT(*) AS trades, SUM(amount_usd) AS volume
FROM trades GROUP BY protocol ORDER BY volume DESC
```

In [ ]:
(df.groupby("protocol")
   .agg(trades=("tx_hash", "count"),            # named aggregation: new_name=(column, function)
        volume_usd=("amount_usd", "sum"),
        avg_trade=("amount_usd", "mean"),
        median_trade=("amount_usd", "median"))
   .sort_values("volume_usd", ascending=False))

The `new_name=(column, function)` form is the cleanest: it reads like `SUM(amount_usd) AS volume_usd`.

### 7.1 Multiple keys, and `reset_index`

Grouping by several columns produces a **MultiIndex**. Call `.reset_index()` to turn the group keys back into normal columns (which is what you want 90% of the time).

In [ ]:
by_proto_pair = (df.groupby(["protocol", "pair"])
                   .agg(volume_usd=("amount_usd", "sum"), trades=("tx_hash", "count"))
                   .reset_index())
by_proto_pair.sort_values("volume_usd", ascending=False).head(6)

### 7.2 Pivot tables: GROUP BY with columns spread out

When you want one grouping key down the side and another across the top (a cross-tab), `pivot_table` does it in one call. In SQL you'd write `SUM(CASE WHEN ...)` per column.

In [ ]:
df.pivot_table(index="protocol", columns="side", values="amount_usd", aggfunc="sum")

### 7.3 `transform`: window functions (`OVER (PARTITION BY ...)`)

`agg` collapses rows. `transform` keeps **every row** and attaches the group's result to each. This is the equivalent of a window function.

```sql
SELECT *, amount_usd / SUM(amount_usd) OVER (PARTITION BY protocol) AS share_of_protocol
```

In [ ]:
df["share_of_protocol"] = df["amount_usd"] / df.groupby("protocol")["amount_usd"].transform("sum")
df[["protocol", "amount_usd", "share_of_protocol"]].head()

In [ ]:
# RANK() OVER (PARTITION BY protocol ORDER BY amount_usd DESC)
df["rank_in_protocol"] = df.groupby("protocol")["amount_usd"].rank(ascending=False, method="first")
df[df["rank_in_protocol"] <= 2].sort_values(["protocol", "rank_in_protocol"])[["protocol", "tx_hash", "amount_usd", "rank_in_protocol"]]

### 7.4 Wallet leaderboard (a very common on-chain query)

In [ ]:
leaderboard = (df.groupby("wallet")
                 .agg(trades=("tx_hash", "count"),
                      volume_usd=("amount_usd", "sum"),
                      gas_spent_usd=("gas_cost_usd", "sum"))
                 .nlargest(5, "volume_usd")
                 .reset_index())
leaderboard["wallet"] = leaderboard["wallet"].map(shorten_address)     # your Week 5 function
leaderboard

## 8. Merge (`JOIN`)

`pd.merge(left, right, on=key, how=...)` is a SQL join. The `how` argument is the join type:

| SQL | Pandas | Keeps |
|---|---|---|
| `INNER JOIN` | `how="inner"` | only keys in both |
| `LEFT JOIN` | `how="left"` | all left rows, NaN where no match |
| `RIGHT JOIN` | `how="right"` | all right rows |
| `FULL OUTER JOIN` | `how="outer"` | everything |

Let's enrich trades with pair metadata and wallet labels.

In [ ]:
pair_info = make_pair_info()
wallet_labels = make_wallet_labels()
print(pair_info.shape, wallet_labels.shape)
pair_info

In [ ]:
# LEFT JOIN pair_info ON trades.pair = pair_info.pair
enriched = df.merge(pair_info, on="pair", how="left", validate="many_to_one")
enriched[["pair", "category", "risk_score", "amount_usd"]].head()

**`validate="many_to_one"` is a seatbelt.** It asserts that the right table has unique keys. If `pair_info` accidentally had two rows for `WETH-USDC`, a plain merge would silently **duplicate** your trades and inflate every volume number. Always validate joins on-chain data, where duplicate keys are common.

### 8.1 LEFT JOIN leaves NULLs: only 25 of 40 wallets are labelled

In [ ]:
enriched = enriched.merge(wallet_labels, on="wallet", how="left")
print("unlabelled trades:", enriched["label"].isna().sum())
enriched["label"] = enriched["label"].fillna("unknown")      # SQL: COALESCE(label, 'unknown')
enriched["label"].value_counts()

### 8.2 Finding what didn't match (anti-join)

`indicator=True` adds a `_merge` column telling you where each row came from. That's how you do `LEFT JOIN ... WHERE right.key IS NULL` (an anti-join):

In [ ]:
check = pair_info.merge(df[["pair"]].drop_duplicates(), on="pair", how="left", indicator=True)
check[check["_merge"] == "left_only"]       # pairs in pair_info with zero trades

### 8.3 Stacking tables: `UNION ALL`

`pd.concat([a, b])` stacks rows (that's how `make_tvl()` built its long table). `ignore_index=True` renumbers the rows.

## 9. NumPy: the engine underneath

Every Pandas column is backed by a **NumPy array**: a fixed-size block of memory holding values of **one type**. That's why maths is fast: no Python objects, no per-element type checks, just C loops over contiguous memory.

In [ ]:
# A Python list of gas prices vs a NumPy array
gas_list = [12.5, 18.0, 25.3, 40.1, 95.0]
gas = np.array(gas_list)

print(type(gas), gas.dtype, gas.shape)
print("list * 2 :", gas_list * 2)       # repeats the list! (not maths)
print("array * 2:", gas * 2)            # element-wise maths

### 9.1 Vectorised maths and broadcasting

Operations apply to every element. **Broadcasting** lets a single number (or smaller array) stretch to match a bigger one, so you never write the loop.

In [ ]:
gas_gwei = np.array([12.5, 18.0, 25.3, 40.1, 95.0])
gas_used = 150_000

cost_eth = gas_gwei * gas_used / 1e9           # scalar broadcast across the array
cost_usd = cost_eth * 3_250
print(np.round(cost_usd, 2))

# 2D broadcasting: cost of every gas price x every tx type
tx_gas = np.array([21_000, 65_000, 150_000, 300_000])       # transfer, ERC20, swap, mint
matrix = gas_gwei[:, None] * tx_gas[None, :] / 1e9 * 3_250   # shape (5, 4)
pd.DataFrame(matrix, index=[f"{g} gwei" for g in gas_gwei],
             columns=["transfer", "erc20", "swap", "mint"])

### 9.2 Statistics you'll use constantly

In [ ]:
prices = df["gas_price_gwei"].dropna().to_numpy()      # Series -> NumPy array

print("mean   :", prices.mean().round(2))
print("median :", np.median(prices).round(2))
print("std    :", prices.std().round(2))
print("p90    :", np.percentile(prices, 90).round(2))
print("p99    :", np.percentile(prices, 99).round(2))

Percentiles are how you answer "what gas price do I need to land in the next block 90% of the time?" Averages hide the spikes that actually hurt users.

### 9.3 Log returns

Analysts prefer **log returns** to simple returns because they add across time: the log return over a week is the sum of the daily log returns. This is the foundation for volatility.

In [ ]:
eth_prices = np.array([3200.0, 3260.0, 3190.0, 3300.0, 3410.0])

simple_returns = eth_prices[1:] / eth_prices[:-1] - 1
log_returns = np.diff(np.log(eth_prices))

print("simple:", np.round(simple_returns * 100, 3), "%")
print("log   :", np.round(log_returns * 100, 3), "%")
print("sum of log returns == total log return:",
      np.isclose(log_returns.sum(), np.log(eth_prices[-1] / eth_prices[0])))

### 9.4 The Wei overflow trap (blockchain-specific, important)

NumPy's default integer is **int64**: max about 9.22 × 10¹⁸. One ETH is 10¹⁸ Wei, so **anything above ~9.2 ETH in Wei does not fit in int64**. Python's own `int` has no limit, which is why your `wei_to_eth` function was always safe.

There are two failure modes, and the second is the dangerous one:

1. **Converting** an oversized value raises an error (loud, easy to catch).
2. **Doing arithmetic** on an int64 array that is *already* near the limit **silently wraps around to a negative number**. No error, no warning.

In [ ]:
print("int64 max :", np.iinfo(np.int64).max)
print("10 ETH Wei:", 10 * 10**18, "(Python int: fine)")

# Failure mode 1: loud
try:
    np.array([10 * 10**18], dtype=np.int64)
except OverflowError as e:
    print("Conversion fails loudly:", e)

# Failure mode 2: SILENT
balance_wei = np.array([5 * 10**18], dtype=np.int64)     # 5 ETH fits...
doubled = balance_wei * 2                                 # ...10 ETH does not
print("5 ETH x 2 =", doubled, "Wei  <- negative balance, no error")

In [ ]:
# What to do with raw Wei columns from the chain:
wei_values = [10 * 10**18, 250 * 10**18, 3 * 10**17]

safe_obj = pd.Series(wei_values, dtype=object)            # keep exact Python ints
as_eth   = safe_obj.map(wei_to_eth)                        # convert BEFORE doing array maths
print(as_eth.tolist())
# Rule: convert Wei -> ETH (float) at the edge, then use fast float arrays.
# For exact accounting on huge values, keep dtype=object or use the decimal module.

## 10. Time series (`date_trunc`, `LAG`, rolling windows)

Blockchain data is time series: gas by hour, TVL by day, volume by week. Pandas treats a **datetime index** as a first-class citizen.

### 10.1 Resample = `date_trunc` + `GROUP BY`

```sql
SELECT date_trunc('day', block_time) AS day, SUM(amount_usd) FROM trades GROUP BY 1
```

In [ ]:
ts = df.set_index("block_time")                       # make the timestamp the index

daily = ts["amount_usd"].resample("D").sum()          # "D" = day, "h" = hour, "W" = week, "ME" = month end
daily.head()

In [ ]:
# Multiple aggregations at once, on a DataFrame
ts.resample("D").agg(volume_usd=("amount_usd", "sum"),
                     trades=("tx_hash", "count"),
                     avg_gas_gwei=("gas_price_gwei", "mean")).head()

### 10.2 Rolling windows (moving averages)

```sql
AVG(base_fee) OVER (ORDER BY timestamp ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)
```

In [ ]:
gas_df = make_gas_prices().set_index("timestamp")
gas_df["ma_6h"]  = gas_df["base_fee_gwei"].rolling(6).mean()           # 6-hour moving average
gas_df["ma_24h"] = gas_df["base_fee_gwei"].rolling(24).mean()          # 24-hour
gas_df.iloc[20:30]

The first rows of a rolling column are `NaN`: there aren't enough prior rows to fill the window yet. Same as SQL window frames at the start of a partition.

### 10.3 `shift` and `pct_change` (`LAG`)

`shift(1)` is `LAG(col, 1)`. `pct_change()` is the same thing as `(col / LAG(col)) - 1`.

In [ ]:
tvl = make_tvl()
aave = tvl[tvl["protocol"] == "Aave"].set_index("date")["tvl_usd"]

out = pd.DataFrame({"tvl_usd": aave})
out["prev_day"]   = out["tvl_usd"].shift(1)                  # LAG(tvl_usd, 1)
out["daily_chg"]  = out["tvl_usd"].pct_change() * 100        # (tvl / LAG(tvl)) - 1
out["wow_chg"]    = out["tvl_usd"].pct_change(7) * 100       # vs 7 days ago
out["cum_max"]    = out["tvl_usd"].cummax()                  # running max (for drawdown)
out["drawdown"]   = (out["tvl_usd"] / out["cum_max"] - 1) * 100
out.tail()

In [ ]:
print("Aave max drawdown over the period:", fmt_pct(out["drawdown"].min()))

### 10.4 Long vs wide TVL

`make_tvl()` is **long** (one row per date per protocol), the shape databases like. Charts and correlations want **wide** (one column per protocol). `pivot` converts:

In [ ]:
wide = tvl.pivot(index="date", columns="protocol", values="tvl_usd")
print(wide.shape)
wide.tail(3)

In [ ]:
# Correlation of daily TVL changes between protocols
wide.pct_change().corr().round(2)

## 11. Mini project: DEX desk report

Put it together. One pipeline: load, clean, enrich, aggregate, report. This is the pattern for every analysis you'll do.

In [ ]:
def dex_report(trades_df: pd.DataFrame, eth_price: float = 3_250.0) -> dict:
    """Summarise a DEX trades table into headline numbers and breakdowns."""
    d = trades_df.copy()
    d["gas_cost_usd"] = d["gas_used"] * d["gas_price_gwei"] / 1e9 * eth_price
    d = d.merge(make_pair_info(), on="pair", how="left", validate="many_to_one")

    return {
        "total_volume":   d["amount_usd"].sum(),
        "trades":         len(d),
        "unique_wallets": d["wallet"].nunique(),                # SQL: COUNT(DISTINCT wallet)
        "median_trade":   d["amount_usd"].median(),
        "p95_gas_gwei":   d["gas_price_gwei"].quantile(0.95),
        "by_protocol":    d.groupby("protocol")["amount_usd"].sum().sort_values(ascending=False),
        "by_category":    d.groupby("category")["amount_usd"].sum().sort_values(ascending=False),
        "busiest_hour":   d["block_time"].dt.hour.value_counts().idxmax(),
    }

r = dex_report(make_dex_trades())
print(f"Volume        : {fmt_large(r['total_volume'])}")
print(f"Trades        : {r['trades']:,}")
print(f"Unique wallets: {r['unique_wallets']}")
print(f"Median trade  : {fmt_usd(r['median_trade'])}")
print(f"p95 gas       : {r['p95_gas_gwei']:.1f} gwei  {gas_tier(r['p95_gas_gwei'])}")
print(f"Busiest hour  : {r['busiest_hour']}:00 UTC")
print()
print(r["by_protocol"].map(fmt_large))

## 12. Common pitfalls

| Pitfall | What happens | Fix |
|---|---|---|
| `df[a > 1 and b < 2]` | `ValueError: truth value ambiguous` | use `&`, `\|` with parentheses |
| Merge on non-unique key | rows silently multiply, volumes inflate | `validate="many_to_one"` |
| `df[df.x > 1]["y"] = 0` | chained assignment: may not modify `df` | `df.loc[df.x > 1, "y"] = 0` |
| Modifying a slice | `SettingWithCopyWarning` / no effect | `.copy()` when you take a subset to modify |
| `apply(axis=1)` on millions of rows | very slow | vectorise with column maths / `np.where` |
| Dates as strings | no `.dt`, no `resample` | `parse_dates=` or `pd.to_datetime()` |
| Wei in int64 | silent overflow | convert to ETH float or use `dtype=object` |
| `mean()` of heavy-tailed data | misleading | also report `median()` and percentiles |

The chained-assignment one deserves a demonstration:

In [ ]:
demo = df[["tx_hash", "amount_usd"]].head(5).copy()

# Correct: single .loc call with row mask and column
demo.loc[demo["amount_usd"] > 1000, "flag"] = "big"
demo["flag"] = demo["flag"].fillna("normal")
demo

## Lesson summary

| Concept | SQL | Pandas |
|---|---|---|
| Inspect | `DESCRIBE`, `COUNT(*)` | `.info()`, `.shape`, `.describe()` |
| Select columns | `SELECT a, b` | `df[["a","b"]]` |
| Filter | `WHERE a > 1 AND b = 'x'` | `df[(df.a > 1) & (df.b == "x")]` / `.query()` |
| IN / BETWEEN / NULL | `IN`, `BETWEEN`, `IS NULL` | `.isin()`, `.between()`, `.isna()` |
| New column | `SELECT a*b AS c` | `df["c"] = df.a * df.b` / `.assign()` |
| CASE WHEN | `CASE WHEN ... END` | `np.where()` / `np.select()` |
| Sort + limit | `ORDER BY ... LIMIT n` | `.sort_values().head(n)` / `.nlargest()` |
| Aggregate | `GROUP BY` | `.groupby().agg(name=(col, func))` |
| Window function | `OVER (PARTITION BY)` | `.groupby().transform()` / `.rank()` |
| Join | `JOIN ... ON` | `.merge(on=, how=, validate=)` |
| UNION ALL | `UNION ALL` | `pd.concat()` |
| date_trunc | `date_trunc('day', t)` | `.resample("D")` / `.dt.date` |
| LAG / moving avg | `LAG()`, `AVG() OVER` | `.shift()`, `.rolling().mean()` |
| NumPy | n/a | vectorised maths, broadcasting, percentiles, log returns |

**Key ideas:** think in whole columns, not rows. Validate every join. Mean lies on heavy-tailed data. Mind Wei vs int64.

## What's next

**Week 9: Data Visualization.** You'll turn these DataFrames into charts: Matplotlib (line, bar, scatter), Seaborn (gas heatmap by hour), and Plotly (interactive TVL charts). Everything you built here (`resample`, `groupby`, `pivot`) is the data prep for those charts.

## Commit your work

```bash
cd ~/Desktop/python-for-blockchain-analytics
git add phase-2-python-for-data/week-08-pandas-numpy/
git commit -m "phase-2/week-08: pandas and numpy — lesson, exercises, solutions"
git push
```

Before you do the exercises, open `exercises.py` and work through them in order. Only open `exercises_solutions.py` after you've attempted them.